In [2]:
# === 0) Imports & global config =================================================
import math, os, json
import numpy as np
import pandas as pd
from pyproj import Geod
import matplotlib.pyplot as plt

# Optional (for sea-only legs and folium maps)
import searoute as sr
import folium

geod = Geod(ellps="WGS84")
os.makedirs("figures", exist_ok=True)
os.makedirs("tables", exist_ok=True)
os.makedirs("maps_variants", exist_ok=True)

In [3]:
# === 1) Helpers =================================================================
def nm_point_to_point(lat1, lon1, lat2, lon2):
    """Great-circle distance in nautical miles between two (lat,lon) points."""
    _, _, dist_m = geod.inv(lon1, lat1, lon2, lat2)
    return dist_m / 1852.0

def gc_chain_distance_nm(latlon_chain):
    """Sum of GC legs over a waypoint chain [(lat,lon), ...]."""
    d = 0.0
    for (la1, lo1), (la2, lo2) in zip(latlon_chain[:-1], latlon_chain[1:]):
        d += nm_point_to_point(la1, lo1, la2, lo2)
    return d


In [4]:
def searoute_leg_coords_and_nm(a_lat, a_lon, b_lat, b_lon):
    """
    Sea-only route between A and B using searoute.
    Returns (coords_lonlat, distance_nm).
    Robust to different searoute return types.
    """
    r = sr.searoute((a_lon, a_lat), (b_lon, b_lat))  # searoute expects (lon,lat)
    # Try to extract coords
    coords = None
    # Case 1: GeoJSON dict with features
    if isinstance(r, dict):
        try:
            feat = r.get("features", [])[0]
            coords = feat["geometry"]["coordinates"]
        except Exception:
            pass
    # Case 2: object with .__geo_interface__
    if coords is None and hasattr(r, "__geo_interface__"):
        gi = r.__geo_interface__
        if "coordinates" in gi:
            coords = gi["coordinates"]
        elif "features" in gi and gi["features"]:
            coords = gi["features"][0]["geometry"]["coordinates"]
    # Case 3: shapely-like LineString
    if coords is None and hasattr(r, "coords"):
        coords = list(r.coords)

    if coords is None:
        raise RuntimeError("Unable to extract searoute coordinates; update extractor.")

    # coords are (lon,lat). Sum segment lengths
    dist_nm = 0.0
    for (lo1, la1), (lo2, la2) in zip(coords[:-1], coords[1:]):
        _, _, dm = geod.inv(lo1, la1, lo2, la2)
        dist_nm += dm / 1852.0
    return coords, dist_nm

def seaonly_chain_distance_nm(latlon_chain):
    """Sum of sea-only legs over a waypoint chain using searoute per leg."""
    total = 0.0
    all_segments = []  # list of list of (lat,lon) for mapping/plots
    for (la1, lo1), (la2, lo2) in zip(latlon_chain[:-1], latlon_chain[1:]):
        coords_lonlat, dleg = searoute_leg_coords_and_nm(la1, lo1, la2, lo2)
        total += dleg
        # convert to (lat,lon)
        seg_latlon = [(la, lo) for (lo, la) in coords_lonlat]
        all_segments.append(seg_latlon)
    return total, all_segments

In [5]:
def hours_from_nm(distance_nm, speed_kn):
    return distance_nm / float(speed_kn)

def fuel_and_co2(hours, me_tpd, aux_tpd, ef_co2=3.114):
    """Fuel (t) = (me_tpd + aux_tpd) * (hours/24); CO2 (t) = fuel_total * EF."""
    me = (me_tpd * hours) / 24.0
    aux = (aux_tpd * hours) / 24.0
    total = me + aux
    co2 = total * ef_co2
    return me, aux, total, co2


In [6]:
# === 2) Scenario definition =====================================================
# Macro waypoints (lat,lon). You can adjust to your earlier waypoint set.
# Origins/Destinations:
ROTTERDAM = (51.95,   4.13)     # N,E
YOKOHAMA  = (35.45, 139.65)

# Common pivots
GIBRALTAR   = (36.0,  -5.6)
PORT_SAID   = (31.3,  32.3)     # north entrance
SUEZ_SOUTH  = (29.9,  32.6)     # south exit into Red Sea
BAB_EL_MANDEB = (12.6, 43.4)
MALACCA     = (1.25, 103.6)

# Cape pivots
OFF_WAFRICA = (20.0, -20.0)
CAPE_GOODHOPE = (-34.9, 18.5)
MID_IND_1   = (-20.0, 60.0)
MID_IND_2   = (-15.0, 90.0)
SUNDA_GATE  = (-6.0, 106.0)
SCS_PIVOT   = (15.0, 120.0)
RYUKYU_SE   = (25.0, 130.0)

# NSR pivots (coarse, channel-guided)
MURMANSK   = (68.9, 33.1)
KARA_GATE  = (70.5, 58.0)
VILKITSKY  = (77.5, 104.0)
LAPTEV     = (75.0, 130.0)
ESS_SEA    = (72.0, 160.0)
BERING     = (66.0, -169.0)    # 191E
ALEUTIANS  = (50.0, 170.0)

# Corridor speeds (kn) and baseline consumption (tonnes per day)
PARAMS = {
    "SUEZ": {"speed_kn": 14.5, "me_tpd": 44.0, "aux_tpd": 4.9},
    "CAPE": {"speed_kn": 14.5, "me_tpd": 44.0, "aux_tpd": 4.9},
    "NSR" : {"speed_kn": 12.5, "me_tpd": 28.0, "aux_tpd": 4.6},
}


In [7]:
# Variants (each is a waypoint chain)
WPT = {
    # --- SUEZ variants ---
    "SUEZ_VAR1": [ROTTERDAM, GIBRALTAR, PORT_SAID, SUEZ_SOUTH, BAB_EL_MANDEB, MALACCA, YOKOHAMA],
    "SUEZ_VAR2": [ROTTERDAM, (37.5,-10.0), PORT_SAID, SUEZ_SOUTH, (16.0,50.0), MALACCA, YOKOHAMA],
    "SUEZ_VAR3": [ROTTERDAM, (34.0,-15.0), PORT_SAID, SUEZ_SOUTH, (14.0,48.0), (8.0,98.0), YOKOHAMA],

    # --- CAPE variants ---
    "CAPE_VAR1": [ROTTERDAM, OFF_WAFRICA, CAPE_GOODHOPE, MID_IND_1, MID_IND_2, SUNDA_GATE, SCS_PIVOT, RYUKYU_SE, YOKOHAMA],
    "CAPE_VAR2": [ROTTERDAM, (25.0,-15.0), CAPE_GOODHOPE, (-30.0,60.0), (-20.0,90.0), (0.0,115.0), (15.0,130.0), YOKOHAMA],
    "CAPE_VAR3": [ROTTERDAM, (30.0,-20.0), CAPE_GOODHOPE, (-25.0,70.0), (-10.0,100.0), (10.0,125.0), (22.0,135.0), YOKOHAMA],

    # --- NSR variants ---
    "NSR_VAR1": [ROTTERDAM, MURMANSK, KARA_GATE, VILKITSKY, LAPTEV, ESS_SEA, BERING, ALEUTIANS, YOKOHAMA],
    "NSR_VAR2": [ROTTERDAM, (66.0,20.0), KARA_GATE, (76.0,95.0), LAPTEV, (73.0,150.0), BERING, (48.0,175.0), YOKOHAMA],
    "NSR_VAR3": [ROTTERDAM, (64.0,15.0), (71.0,45.0), (76.0,90.0), (75.0,135.0), (71.0,165.0), BERING, (49.0,172.0), YOKOHAMA],
}

In [11]:
def scenario_meta(key):
    if key.startswith("SUEZ"): return "SUEZ"
    if key.startswith("CAPE"): return "CAPE"
    if key.startswith("NSR"):  return "NSR"
    return "UNKNOWN"

# === 3) Compute master results ==================================================
rows = []
routes_cache = {}  # for optional folium maps

for key, chain in WPT.items():
    corridor = scenario_meta(key)
    speed_kn = PARAMS[corridor]["speed_kn"]
    me_tpd   = PARAMS[corridor]["me_tpd"]
    aux_tpd  = PARAMS[corridor]["aux_tpd"]

    # Distances
    d_gc_via = gc_chain_distance_nm(chain)
    d_sea, segs = seaonly_chain_distance_nm(chain)
    routes_cache[key] = segs  # list of segments for mapping

    # Time / Fuel / CO2
    h = hours_from_nm(d_sea, speed_kn)
    me_t, aux_t, fuel_t, co2_t = fuel_and_co2(h, me_tpd, aux_tpd, ef_co2=3.114)

    rows.append({
        "scenario": key,
        "corridor": corridor,
        "speed_kn": speed_kn,
        "distance_gc_nm": d_gc_via,
        "distance_sea_nm": d_sea,
        "delta_nm": d_sea - d_gc_via,
        "delta_pct": 100.0 * (d_sea - d_gc_via) / d_gc_via,
        "hours": h,
        "days": h / 24.0,
        "fuel_me_t": me_t,
        "fuel_aux_t": aux_t,
        "fuel_total_t": fuel_t,
        "co2_t": co2_t,
        "legs": len(chain) - 1
    })

df = pd.DataFrame(rows).sort_values(["corridor","scenario"]).reset_index(drop=True)
df.to_csv("tables/results_master.csv", index=False)

RuntimeError: Unable to extract searoute coordinates; update extractor.

In [ ]:




# === 4) Corridor-level summaries ===============================================
def summarize_corridor(df, corridor):
    sub = df[df["corridor"]==corridor]
    agg = sub.agg({
        "distance_sea_nm":["min","median","max"],
        "hours":["min","median","max"],
        "fuel_total_t":["min","median","max"],
        "co2_t":["min","median","max"],
    }).round(2).T
    agg.columns = ["min","median","max"]
    return agg

summaries = {c: summarize_corridor(df, c) for c in ["NSR","SUEZ","CAPE"]}
for c, tab in summaries.items():
    tab.to_csv(f"tables/summary_{c}.csv")

# === 5) Tables (Markdown and CSV) ==============================================
# Table 1: distance comparison
tbl1 = df[["scenario","corridor","distance_gc_nm","distance_sea_nm","delta_nm","delta_pct"]].copy()
tbl1 = tbl1.round({"distance_gc_nm":0,"distance_sea_nm":0,"delta_nm":0,"delta_pct":2})
tbl1.to_csv("tables/table1_distance.csv", index=False)

# Table 2: time comparison
tbl2 = df[["scenario","corridor","speed_kn","hours","days"]].copy().round(2)
tbl2.to_csv("tables/table2_time.csv", index=False)

# Table 3: fuel & CO2
tbl3 = df[["scenario","corridor","fuel_me_t","fuel_aux_t","fuel_total_t","co2_t"]].copy().round(1)
tbl3.to_csv("tables/table3_fuel_co2.csv", index=False)

# Table 4: variant sensitivity
rows_sens = []
for c in ["NSR","SUEZ","CAPE"]:
    sub = df[df["corridor"]==c]
    for metric in ["distance_sea_nm","hours","fuel_total_t","co2_t"]:
        rows_sens.append({
            "corridor": c,
            "metric": metric,
            "min": sub[metric].min(),
            "median": sub[metric].median(),
            "max": sub[metric].max(),
            "spread_pct": 100.0 * (sub[metric].max() - sub[metric].min()) / sub[metric].median()
        })
tbl4 = pd.DataFrame(rows_sens).round(2)
tbl4.to_csv("tables/table4_variant_sensitivity.csv", index=False)

# === 6) Figures (matplotlib; no seaborn; one plot each; no custom colors) ======
plt.figure()
ax = df.sort_values("distance_sea_nm").plot(
    kind="bar", x="scenario", y="distance_sea_nm", legend=False, rot=45)
ax.set_ylabel("Sea-only distance (nm)")
ax.set_title("Figure 1. Sea-only distance by scenario")
plt.tight_layout()
plt.savefig("figures/fig1_distance.png", dpi=200)
plt.close()

plt.figure()
ax = df.sort_values("hours").plot(
    kind="bar", x="scenario", y="hours", legend=False, rot=45)
ax.set_ylabel("Voyage time (hours)")
ax.set_title("Figure 2. Hours by scenario (corridor speeds)")
plt.tight_layout()
plt.savefig("figures/fig2_time.png", dpi=200)
plt.close()

plt.figure()
# Stacked bars for fuel (main+aux)
fuel = df[["fuel_me_t","fuel_aux_t"]].copy()
fuel.index = df["scenario"]
fuel.plot(kind="bar", stacked=True, legend=True, rot=45)
plt.ylabel("Fuel (t)")
plt.title("Figure 3. Fuel by scenario (main + auxiliary)")
plt.tight_layout()
plt.savefig("figures/fig3_fuel_stacked.png", dpi=200)
plt.close()

# Variant spread by corridor (distance)
plt.figure()
for c in ["NSR","SUEZ","CAPE"]:
    sub = df[df["corridor"]==c]
    xs = np.arange(len(sub))
    plt.plot(xs, sub["distance_sea_nm"].values, marker="o", label=c)
plt.xticks([])
plt.ylabel("Sea-only distance (nm)")
plt.title("Figure 4. Variant spread by corridor")
plt.legend()
plt.tight_layout()
plt.savefig("figures/fig4_variant_spread.png", dpi=200)
plt.close()

# === 7) Optional: folium maps for each variant =================================
def build_variant_map(key, out_dir="maps_variants"):
    segs = routes_cache.get(key, [])
    # center map roughly at mid-point of all coords
    all_pts = [pt for seg in segs for pt in seg]
    lat_c = sum(p[0] for p in all_pts)/len(all_pts)
    lon_c = sum(p[1] for p in all_pts)/len(all_pts)
    m = folium.Map(location=[lat_c, lon_c], zoom_start=3, tiles="CartoDB positron")
    # draw segments
    for seg in segs:
        folium.PolyLine(locations=seg, weight=3, opacity=0.9).add_to(m)
    folium.LayerControl().add_to(m)
    m.save(os.path.join(out_dir, f"{key}.html"))

for key in WPT.keys():
    try:
        build_variant_map(key)
    except Exception as e:
        print(f"[map] {key}: {e}")

print("Done. See ./tables and ./figures (and ./maps_variants).")
